# Data Modeling: Sales & Payments Lakehouse Project

**Author:** Artem Liakh

**Role:** Product / Data Analyst

**Description:**

Data ingestion, quality validation, transformation and analytical
modeling of sales and payment data in Databricks Lakehouse.

---

## Project Objective

Build an analytical data model from raw CRM/POS exports and
transform denormalized source files into a dimensional model
suitable for analytical queries and BI reporting.

---

## Source Data

The project uses two raw CSV files:

- `sales.csv` — one row represents one product within an order.
- `payments.csv` — one row represents one payment transaction.

Source files are stored in a Unity Catalog Volume:

`/Volumes/data_lab/shop/raw/`

---

## Target Data Model

The final analytical model will contain:

### Fact tables

- `fact_sales`
- `fact_payments`

### Dimension tables

- `dim_customer`
- `dim_product`
- `dim_date`

The model will use shared/conformed dimensions where appropriate.

---

## Data Modeling Principles

The transformation will demonstrate:

1. Raw file ingestion
2. Schema inspection
3. Data profiling
4. Data quality validation
5. Identification of business keys
6. Identification of table grain
7. Dimension construction
8. Fact table construction
9. Referential integrity validation
10. Analytical model validation

---

## Business Questions

The resulting model should support analysis such as:

- Sales by date
- Sales by customer
- Sales by product
- Sales by category
- Payments by date
- Payments by payment method
- Sales and payments by customer
- Sales and payment activity over time

---

## Architecture

Raw CSV files:

`CSV → Unity Catalog Volume → Spark DataFrame`

Analytical transformation:

`Raw Data → Data Quality Checks → Dimensions + Facts`

Target model:

`Star Schema / Dimensional Model`

---

## Environment

**Platform:** Databricks

**Processing Engine:** Apache Spark / PySpark

**Storage Layer:** Delta Lake

**Governance:** Unity Catalog

**Source Format:** CSV

**Target Format:** Delta Tables

In [0]:
# ============================================================
# Step 1. Define source file paths
#
# Objective:
# Define explicit paths to the raw CSV files stored in the
# Unity Catalog Volume.
#
# The files represent the raw source layer of the project.
# At this stage, the source data is not transformed or modified.
# ============================================================

sales_path = "/Volumes/data_lab/shop/raw/sales.csv"

payments_path = "/Volumes/data_lab/shop/raw/payments.csv"

In [0]:
# ============================================================
# Step 2. Load raw CSV files
#
# Objective:
# Read the raw CSV files from the Unity Catalog Volume into
# Spark DataFrames.
#
# No business transformations are performed at this stage.
# The purpose is to establish the connection between the raw
# file layer and the Spark processing layer.
#
# Parameters:
#
# header=True
# The first row of each CSV contains column names.
#
# inferSchema=True
# Spark attempts to infer the data type of each column.
# ============================================================

sales_df = spark.read.csv(
    sales_path,
    header=True,
    inferSchema=True
)

payments_df = spark.read.csv(
    payments_path,
    header=True,
    inferSchema=True
)

## Step 3. Initial Data Inspection

### Objective

Verify that both source datasets were successfully loaded into
Spark DataFrames and inspect their initial structure.

### Validation scope

1. Display sample records.
2. Inspect column names.
3. Inspect inferred data types.
4. Verify that the source files are readable.

### Principle

At this stage, the raw data is only inspected.

No records are removed, modified or aggregated before the
data quality assessment is completed.

In [0]:
# ============================================================
# Step 3.1. Preview sales data
#
# Objective:
# Display the loaded sales dataset to verify that the CSV was
# read correctly and that the expected columns are present.
# ============================================================

display(sales_df)

order_id,order_date,customer_id,customer_name,city,product_id,product_name,category,quantity,unit_price
1000,2026-07-21,3,Марія Бондаренко,Одеса,101,Ноутбук Air 13,Ноутбуки,1,32000
1000,2026-07-21,3,Марія Бондаренко,Одеса,102,Ноутбук Pro 14,Ноутбуки,1,48000
1001,2026-07-03,2,Андрій Коваленко,Львів,107,Кабель USB-C 1м,Аксесуари,1,350
1001,2026-07-03,2,Андрій Коваленко,Львів,102,Ноутбук Pro 14,Ноутбуки,1,48000
1002,2026-07-04,10,Максим Савченко,Одеса,104,Смартфон M5,Смартфони,1,12500
1003,2026-07-03,9,Ірина Лисенко,Вінниця,105,Навушники Beat,Навушники,1,3200
1004,2026-08-06,5,Наталія Мельник,Дніпро,103,Смартфон S10,Смартфони,1,21000
1004,2026-08-06,5,Наталія Мельник,Дніпро,102,Ноутбук Pro 14,Ноутбуки,1,48000
1004,2026-08-06,5,Наталія Мельник,Дніпро,104,Смартфон M5,Смартфони,1,12500
1005,2026-08-06,1,Олена Шевченко,Київ,104,Смартфон M5,Смартфони,1,12500


In [0]:
# ============================================================
# Step 3.2. Preview payments data
#
# Objective:
# Display the loaded payments dataset to verify that the CSV
# was read correctly and that the expected columns are present.
# ============================================================

display(payments_df)

payment_id,order_id,payment_date,customer_id,customer_name,city,method,amount
5000,1000,2026-07-21,3,Марія Бондаренко,Одеса,картка,24000
5001,1000,2026-08-09,3,Марія Бондаренко,Одеса,картка,56000
5002,1001,2026-07-03,2,Андрій Коваленко,Львів,готівка,48350
5003,1002,2026-07-04,10,Максим Савченко,Одеса,готівка,3750
5004,1002,2026-07-08,10,Максим Савченко,Одеса,картка,8750
5005,1003,2026-07-03,9,Ірина Лисенко,Вінниця,картка,3200
5006,1004,2026-08-06,5,Наталія Мельник,Дніпро,картка,81500
5007,1005,2026-08-06,1,Олена Шевченко,Київ,готівка,14050
5008,1006,2026-07-24,5,Наталія Мельник,Дніпро,Apple Pay,6300
5009,1008,2026-07-27,3,Марія Бондаренко,Одеса,картка,22200


## Step 4. Source Schema Validation

### Objective

Inspect the Spark schema inferred from the raw CSV files.

Correct data types are important because they determine how
the data can be filtered, joined, aggregated and transformed.

Particular attention will be paid to:

- identifiers
- dates
- numeric measures
- categorical attributes

In [0]:
# ============================================================
# Step 4.1. Inspect sales schema
#
# Objective:
# Review column names and inferred data types for the sales
# dataset.
# ============================================================

sales_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)



In [0]:
# ============================================================
# Step 4.2. Inspect payments schema
#
# Objective:
# Review column names and inferred data types for the payments
# dataset.
# ============================================================

payments_df.printSchema()

root
 |-- payment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- method: string (nullable = true)
 |-- amount: integer (nullable = true)



## Step 5. Dimensions

### Objective

Create dimension tables from the source datasets.

Dimensions are reference tables that describe business entities
used for analytical analysis.

The project contains two dimensions at this stage:

- `dim_customer` — one row per unique customer.
- `dim_product` — one row per unique product.

### Customer dimension

`dim_customer` is a shared dimension for both fact tables.

Customer information is extracted from both `sales` and `payments`
because a customer may appear in either source dataset.

The datasets are combined using `union()` and duplicate records
are removed using `distinct()`.

### Product dimension

`dim_product` is specific to the sales process.

Products are extracted only from `sales` because the payments
dataset does not contain product-level information.

### Ordering

`orderBy()` is used only to make the resulting DataFrame easier
to inspect.

It does not define the logical order of records in the final
analytical model.

In [0]:
# ============================================================
# Step 5.1. Create customer dimension
#
# Objective:
# Create a shared customer dimension from both source datasets.
#
# Why both sales and payments?
# Customer information exists in both datasets. Combining both
# sources ensures that the dimension contains customers found
# in either source.
#
# union():
# Combines rows from the sales and payments datasets.
#
# distinct():
# Removes duplicate customer records.
#
# orderBy():
# Sorts the result by customer_id to make the data easier
# to inspect.
# ============================================================

dim_customer = (
    sales_df.select(
        "customer_id",
        "customer_name",
        "city"
    )
    .union(
        payments_df.select(
            "customer_id",
            "customer_name",
            "city"
        )
    )
    .distinct()
    .orderBy("customer_id")
)

display(dim_customer)

customer_id,customer_name,city
1,Олена Шевченко,Київ
2,Андрій Коваленко,Львів
3,Марія Бондаренко,Одеса
4,Ігор Ткаченко,Київ
5,Наталія Мельник,Дніпро
6,Дмитро Кравченко,Харків
7,Оксана Олійник,Львів
8,Сергій Поліщук,Київ
9,Ірина Лисенко,Вінниця
10,Максим Савченко,Одеса


In [0]:
# ============================================================
# Step 5.2. Create product dimension
#
# Objective:
# Create a product dimension containing one unique row
# per product.
#
# Source:
# sales_df
#
# distinct():
# Removes duplicate product records because the same product
# can appear in multiple sales order lines.
#
# orderBy():
# Sorts products by product_id for easier inspection.
# ============================================================

dim_product = (
    sales_df.select(
        "product_id",
        "product_name",
        "category"
    )
    .distinct()
    .orderBy("product_id")
)

display(dim_product)

product_id,product_name,category
101,Ноутбук Air 13,Ноутбуки
102,Ноутбук Pro 14,Ноутбуки
103,Смартфон S10,Смартфони
104,Смартфон M5,Смартфони
105,Навушники Beat,Навушники
106,TWS Mini,Навушники
107,Кабель USB-C 1м,Аксесуари
108,Зарядка 65W,Аксесуари
109,SQL для всіх,Книги
110,Моделювання даних,Книги


## Step 6. Calendar Dimension

### Objective

Create a calendar dimension containing one row for each calendar day
within the analytical period.

Unlike customer and product dimensions, the date dimension is not
extracted from the source CSV files.

The calendar is generated independently because it represents a
standard reference dimension for time-based analysis.

### Date range

The calendar covers the period:

- 2026-07-01
- 2026-09-30

### Attributes

The dimension contains:

- `date` — calendar date and primary key of the dimension.
- `month` — month in `YYYY-MM` format.
- `day_name` — name of the day of the week.
- `is_weekend` — boolean indicator showing whether the date is
  Saturday or Sunday.

### Modeling principle

The same `dim_date` will later be connected to both fact tables:

`dim_date → fact_sales`

`dim_date → fact_payments`

Using the date itself as the key keeps the analytical model
simple and easy to read.

In [0]:
# ============================================================
# Step 6.1. Create calendar dimension
#
# Objective:
# Generate one row for every calendar day from 2026-07-01
# through 2026-09-30.
#
# The date dimension is generated independently from the
# source CSV files.
#
# sequence():
# Generates a sequence of dates between the specified start
# and end dates.
#
# explode():
# Converts the generated array of dates into individual rows.
#
# Additional attributes:
# - month      -> month in YYYY-MM format
# - day_name   -> full name of the day of the week
# - is_weekend -> True for Saturday or Sunday
# ============================================================

from pyspark.sql import functions as F

dim_date = (
    spark.sql(
        """
        SELECT explode(
            sequence(
                DATE '2026-07-01',
                DATE '2026-09-30'
            )
        ) AS date
        """
    )
    .withColumn(
        "month",
        F.date_format("date", "yyyy-MM")
    )
    .withColumn(
        "day_name",
        F.date_format("date", "EEEE")
    )
    .withColumn(
        "is_weekend",
        F.dayofweek("date").isin(1, 7)
    )
)

display(dim_date.limit(7))

date,month,day_name,is_weekend
2026-07-01,2026-07,Wednesday,false
2026-07-02,2026-07,Thursday,false
2026-07-03,2026-07,Friday,false
2026-07-04,2026-07,Saturday,true
2026-07-05,2026-07,Sunday,true
2026-07-06,2026-07,Monday,false
2026-07-07,2026-07,Tuesday,false


In [0]:
# ============================================================
# Step 6.2. Validate calendar dimension
#
# Objective:
# Verify the number of generated dates.
#
# The period from 2026-07-01 through 2026-09-30 is inclusive.
# Therefore, the expected number of rows is 92 days.
# ============================================================

dim_date.count()

92

## Step 7. Fact Tables

### Objective

Create two fact tables representing two different business
processes:

- `fact_sales` — sales transactions.
- `fact_payments` — payment transactions.

Each fact table has its own grain.

### Fact table grain

`fact_sales`:

One row represents one product line within one order.

Example:

One order containing 3 different products produces 3 rows
in `fact_sales`.

Measures:

- `quantity`
- `unit_price`
- `amount`

`fact_payments`:

One row represents one payment transaction.

One order may contain multiple payments, therefore `order_id`
is not unique in `fact_payments`.

Measure:

- `amount`

### Modeling principle

Fact tables contain:

- keys used to connect dimensions;
- numeric measures used for aggregation.

Descriptive attributes such as customer name, city, product name
and category are stored in dimension tables rather than repeated
inside the fact tables.

### Important relationship principle

Both fact tables contain `order_id`, but they must not be joined
directly row-to-row for analytical aggregation.

For example:

2 products × 2 payments = 4 joined rows.

Such a join can multiply the original sales and payment amounts
and produce incorrect totals.

The two fact tables are therefore analyzed through shared
dimensions such as `dim_customer` and `dim_date`.

In [0]:
# ============================================================
# Step 7.1. Create sales fact table
#
# Objective:
# Create a fact table representing individual sales order lines.
#
# Grain:
# One row = one product within one order.
#
# Keys:
# - order_id
# - date
# - customer_id
# - product_id
#
# Measures:
# - quantity
# - unit_price
# - amount
#
# amount is calculated as:
# quantity × unit_price
#
# Descriptive attributes such as customer_name, city,
# product_name and category are intentionally excluded because
# they belong to dimension tables.
# ============================================================

fact_sales = sales_df.select(
    "order_id",
    F.col("order_date").alias("date"),
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    (
        F.col("quantity") * F.col("unit_price")
    ).alias("amount"),
)

display(fact_sales.limit(10))

order_id,date,customer_id,product_id,quantity,unit_price,amount
1000,2026-07-21,3,101,1,32000,32000
1000,2026-07-21,3,102,1,48000,48000
1001,2026-07-03,2,107,1,350,350
1001,2026-07-03,2,102,1,48000,48000
1002,2026-07-04,10,104,1,12500,12500
1003,2026-07-03,9,105,1,3200,3200
1004,2026-08-06,5,103,1,21000,21000
1004,2026-08-06,5,102,1,48000,48000
1004,2026-08-06,5,104,1,12500,12500
1005,2026-08-06,1,104,1,12500,12500


In [0]:
# ============================================================
# Step 7.2. Create payments fact table
#
# Objective:
# Create a fact table representing individual payment
# transactions.
#
# Grain:
# One row = one payment.
#
# Keys:
# - payment_id
# - order_id
# - date
# - customer_id
#
# Additional dimension key:
# - method
#
# Measure:
# - amount
#
# Payment amount is taken directly from the source dataset.
#
# Customer descriptive attributes are excluded because they are
# stored in dim_customer.
# ============================================================

fact_payments = payments_df.select(
    "payment_id",
    "order_id",
    F.col("payment_date").alias("date"),
    "customer_id",
    "method",
    "amount",
)

display(fact_payments.limit(10))

payment_id,order_id,date,customer_id,method,amount
5000,1000,2026-07-21,3,картка,24000
5001,1000,2026-08-09,3,картка,56000
5002,1001,2026-07-03,2,готівка,48350
5003,1002,2026-07-04,10,готівка,3750
5004,1002,2026-07-08,10,картка,8750
5005,1003,2026-07-03,9,картка,3200
5006,1004,2026-08-06,5,картка,81500
5007,1005,2026-08-06,1,готівка,14050
5008,1006,2026-07-24,5,Apple Pay,6300
5009,1008,2026-07-27,3,картка,22200


## Step 8. Save Analytical Model to the Lakehouse

### Objective

Persist the dimension and fact DataFrames as Delta tables
in the Unity Catalog.

Until this step, the objects created in the Notebook are
Spark DataFrames used for processing and transformation.

The `saveAsTable()` operation stores the DataFrames as
persistent tables that can later be queried using SQL and
used by dashboards or other notebooks.

### Target schema

All analytical tables will be stored in:

`data_lab.shop`

### Tables

Dimension tables:

- `dim_customer`
- `dim_product`
- `dim_date`

Fact tables:

- `fact_sales`
- `fact_payments`

### Storage format

The DataFrames are saved as Delta tables.

Delta Lake provides a reliable table layer for analytical
data stored in the Lakehouse.

### Write mode

`overwrite` is used because this Notebook builds the
analytical tables from the source data from scratch.

If the tables already exist, their previous contents will
be replaced by the newly generated DataFrames.

In [0]:
# ============================================================
# Step 8.1. Save dimension tables
#
# Objective:
# Persist the three dimension DataFrames as Delta tables
# in the Unity Catalog schema data_lab.shop.
#
# Target tables:
# - data_lab.shop.dim_customer
# - data_lab.shop.dim_product
# - data_lab.shop.dim_date
#
# mode("overwrite"):
# Replace an existing table with the current DataFrame.
# This is appropriate while rebuilding the analytical model
# during development.
# ============================================================

dim_customer.write \
    .mode("overwrite") \
    .saveAsTable("data_lab.shop.dim_customer")

dim_product.write \
    .mode("overwrite") \
    .saveAsTable("data_lab.shop.dim_product")

dim_date.write \
    .mode("overwrite") \
    .saveAsTable("data_lab.shop.dim_date")

In [0]:
# ============================================================
# Step 8.2. Save fact tables
#
# Objective:
# Persist the two fact DataFrames as Delta tables
# in the Unity Catalog schema data_lab.shop.
#
# Target tables:
# - data_lab.shop.fact_sales
# - data_lab.shop.fact_payments
#
# The fact tables preserve their original business grain:
#
# fact_sales:
# one product line within one order
#
# fact_payments:
# one payment transaction
# ============================================================

fact_sales.write \
    .mode("overwrite") \
    .saveAsTable("data_lab.shop.fact_sales")

fact_payments.write \
    .mode("overwrite") \
    .saveAsTable("data_lab.shop.fact_payments")

In [0]:
# ============================================================
# Step 8.3. Verify Lakehouse tables
#
# Objective:
# Display all tables registered in the data_lab.shop schema.
#
# SHOW TABLES confirms that the DataFrames were successfully
# persisted as catalog tables.
# ============================================================

display(
    spark.sql(
        "SHOW TABLES IN data_lab.shop"
    )
)

database,tableName,isTemporary
shop,dim_customer,false
shop,dim_date,false
shop,dim_product,false
shop,fact_payments,false
shop,fact_sales,false


## Step 9. Analytical Model Validation

### Objective

Validate the analytical model after saving the DataFrames
as Delta tables.

The validation focuses on:

1. Table existence.
2. Table structure.
3. Row counts.
4. Fact table grain.
5. Key uniqueness.
6. Referential integrity.
7. Prevention of fact-to-fact fan-out.

### Validation principle

Creating a table does not guarantee that the underlying
data model is correct.

Each fact table must preserve its intended grain, and every
foreign key used by a fact table should have a corresponding
record in the appropriate dimension.

The two fact tables will also be validated separately because
they represent different business processes.

In [0]:
# ============================================================
# Step 9.1. Verify analytical tables
#
# Objective:
# Confirm that all five analytical tables exist in the
# data_lab.shop schema.
#
# Expected tables:
# - dim_customer
# - dim_product
# - dim_date
# - fact_sales
# - fact_payments
# ============================================================

display(
    spark.sql("""
        SHOW TABLES IN data_lab.shop
    """)
)

database,tableName,isTemporary
shop,dim_customer,false
shop,dim_date,false
shop,dim_product,false
shop,fact_payments,false
shop,fact_sales,false


In [0]:
# ============================================================
# Step 9.2. Inspect fact_sales schema
#
# Objective:
# Verify that fact_sales contains only the keys and measures
# defined by the analytical model.
# ============================================================

display(
    spark.sql("""
        DESCRIBE data_lab.shop.fact_sales
    """)
)

col_name,data_type,comment
order_id,int,null
date,date,null
customer_id,int,null
product_id,int,null
quantity,int,null
unit_price,int,null
amount,int,null


In [0]:
# ============================================================
# Step 9.3. Inspect fact_payments schema
#
# Objective:
# Verify the structure of the payments fact table.
# ============================================================

display(
    spark.sql("""
        DESCRIBE data_lab.shop.fact_payments
    """)
)

col_name,data_type,comment
payment_id,int,null
order_id,int,null
date,date,null
customer_id,int,null
method,string,null
amount,int,null


### Step 9.4. Row Count Validation

The number of rows in each table is checked to understand
the size of the resulting dimensions and facts.

For fact tables, the row count should correspond to their
business grain:

- `fact_sales` — number of product lines.
- `fact_payments` — number of payment transactions.

For dimensions, the row count represents the number of
unique business entities or calendar dates.

In [0]:
# ============================================================
# Step 9.4. Count rows in all analytical tables
#
# Objective:
# Compare the size of dimensions and fact tables.
# ============================================================

display(
    spark.sql("""
        SELECT
            (SELECT COUNT(*) FROM data_lab.shop.dim_customer)  AS customers,
            (SELECT COUNT(*) FROM data_lab.shop.dim_product)   AS products,
            (SELECT COUNT(*) FROM data_lab.shop.dim_date)      AS dates,
            (SELECT COUNT(*) FROM data_lab.shop.fact_sales)    AS sales_rows,
            (SELECT COUNT(*) FROM data_lab.shop.fact_payments) AS payment_rows
    """)
)

customers,products,dates,sales_rows,payment_rows
12,10,92,122,74


In [0]:
# ============================================================
# Step 9.5. Validate fact_sales grain
#
# Objective:
# Verify that the combination of order_id and product_id
# uniquely identifies a sales line.
#
# Expected result:
# Zero rows.
#
# If a combination appears more than once, the assumed grain
# of fact_sales does not hold for the source data.
# ============================================================

display(
    spark.sql("""
        SELECT
            order_id,
            product_id,
            COUNT(*) AS row_count
        FROM data_lab.shop.fact_sales
        GROUP BY
            order_id,
            product_id
        HAVING COUNT(*) > 1
        ORDER BY row_count DESC
    """)
)

order_id,product_id,row_count


In [0]:
# ============================================================
# Step 9.6. Validate fact_payments grain
#
# Objective:
# Verify that payment_id uniquely identifies one payment.
#
# Expected result:
# Zero rows.
# ============================================================

display(
    spark.sql("""
        SELECT
            payment_id,
            COUNT(*) AS row_count
        FROM data_lab.shop.fact_payments
        GROUP BY payment_id
        HAVING COUNT(*) > 1
        ORDER BY row_count DESC
    """)
)

payment_id,row_count


In [0]:
# ============================================================
# Step 9.7. Validate fact_sales → dim_customer relationship
#
# Objective:
# Find sales records whose customer_id does not exist
# in dim_customer.
#
# Expected result:
# Zero rows.
#
# Such records would represent orphan foreign keys.
# ============================================================

display(
    spark.sql("""
        SELECT DISTINCT
            f.customer_id
        FROM data_lab.shop.fact_sales f
        LEFT JOIN data_lab.shop.dim_customer d
            ON f.customer_id = d.customer_id
        WHERE d.customer_id IS NULL
    """)
)

customer_id


In [0]:
# ============================================================
# Step 9.8. Validate fact_sales → dim_product relationship
#
# Objective:
# Find sales records whose product_id does not exist
# in dim_product.
#
# Expected result:
# Zero rows.
# ============================================================

display(
    spark.sql("""
        SELECT DISTINCT
            f.product_id
        FROM data_lab.shop.fact_sales f
        LEFT JOIN data_lab.shop.dim_product d
            ON f.product_id = d.product_id
        WHERE d.product_id IS NULL
    """)
)

product_id


In [0]:
# ============================================================
# Step 9.9. Validate fact_payments → dim_customer relationship
#
# Objective:
# Find payment records whose customer_id does not exist
# in dim_customer.
#
# Expected result:
# Zero rows.
# ============================================================

display(
    spark.sql("""
        SELECT DISTINCT
            f.customer_id
        FROM data_lab.shop.fact_payments f
        LEFT JOIN data_lab.shop.dim_customer d
            ON f.customer_id = d.customer_id
        WHERE d.customer_id IS NULL
    """)
)

customer_id


In [0]:
# ============================================================
# Step 9.10. Validate fact_sales → dim_date relationship
#
# Objective:
# Verify that every sales date exists in the calendar dimension.
#
# Expected result:
# Zero rows.
# ============================================================

display(
    spark.sql("""
        SELECT DISTINCT
            f.date
        FROM data_lab.shop.fact_sales f
        LEFT JOIN data_lab.shop.dim_date d
            ON f.date = d.date
        WHERE d.date IS NULL
        ORDER BY f.date
    """)
)

date


In [0]:
# ============================================================
# Step 9.11. Validate fact_payments → dim_date relationship
#
# Objective:
# Verify that every payment date exists in the calendar dimension.
#
# Expected result:
# Zero rows.
# ============================================================

display(
    spark.sql("""
        SELECT DISTINCT
            f.date
        FROM data_lab.shop.fact_payments f
        LEFT JOIN data_lab.shop.dim_date d
            ON f.date = d.date
        WHERE d.date IS NULL
        ORDER BY f.date
    """)
)

date


### Step 9.12. Fact-to-Fact Fan-out Demonstration

The two fact tables share `order_id`, but they have different
grains.

Joining them directly on `order_id` can multiply rows.

This validation demonstrates why sales and payments should be
aggregated independently before any comparison at order level.

In [0]:
# ============================================================
# Step 9.12. Compare fact table grain by order
#
# Objective:
# Show how many sales lines and payments can belong to the
# same order.
#
# This helps demonstrate why a direct fact-to-fact join can
# multiply rows.
# ============================================================

display(
    spark.sql("""
        SELECT
            s.order_id,
            COUNT(*) AS sales_lines,
            COALESCE(p.payment_count, 0) AS payment_count
        FROM data_lab.shop.fact_sales s
        LEFT JOIN (
            SELECT
                order_id,
                COUNT(*) AS payment_count
            FROM data_lab.shop.fact_payments
            GROUP BY order_id
        ) p
            ON s.order_id = p.order_id
        GROUP BY
            s.order_id,
            p.payment_count
        ORDER BY
            sales_lines DESC,
            payment_count DESC
        LIMIT 20
    """)
)

order_id,sales_lines,payment_count
1020,3,3
1031,3,3
1022,3,2
1011,3,2
1037,3,2
1009,3,2
1005,3,1
1024,3,1
1010,3,1
1032,3,1


## Step 10. Sales Analysis

### Objective

Analyze sales revenue by:

- city
- month
- product category

### Analytical principle

The measure `amount` is taken from `fact_sales`.

Descriptive attributes are obtained from dimensions:

- `dim_customer` → city
- `dim_date` → month
- `dim_product` → category

The fact table is therefore used for measures, while dimensions
provide the analytical dimensions used for grouping.

### Expected output

One row represents one combination of:

`city + month + category`

`revenue` represents the total sales amount for that combination.

In [0]:
%sql
-- ============================================================
-- Step 10.1. Revenue by city, month and product category
--
-- Objective:
-- Calculate total sales revenue for each combination of
-- city, month and product category.
--
-- fact_sales:
-- Provides the sales measure: amount.
--
-- dim_customer:
-- Provides city.
--
-- dim_date:
-- Provides month.
--
-- dim_product:
-- Provides product category.
-- ============================================================

SELECT
    c.city,
    d.month,
    p.category,
    SUM(f.amount) AS revenue

FROM data_lab.shop.fact_sales f

JOIN data_lab.shop.dim_date d
    ON d.date = f.date

JOIN data_lab.shop.dim_customer c
    ON c.customer_id = f.customer_id

JOIN data_lab.shop.dim_product p
    ON p.product_id = f.product_id

GROUP BY
    c.city,
    d.month,
    p.category

ORDER BY
    d.month,
    c.city,
    p.category;

city,month,category,revenue
Вінниця,2026-07,Навушники,3200
Дніпро,2026-07,Книги,1200
Дніпро,2026-07,Ноутбуки,112000
Дніпро,2026-07,Смартфони,75500
Київ,2026-07,Аксесуари,3950
Київ,2026-07,Книги,3750
Київ,2026-07,Навушники,15300
Київ,2026-07,Ноутбуки,128000
Київ,2026-07,Смартфони,21000
Львів,2026-07,Аксесуари,700


In [0]:
%sql

-- ============================================================
-- Step 10.2. Payments by month and payment method
--
-- Objective:
-- Calculate the total amount paid and the number of payment
-- transactions for each payment method by month.
--
-- fact_payments:
-- Provides the payment measure `amount` and payment method
-- `method`.
--
-- dim_date:
-- Provides the calendar attribute `month`.
--
-- Analytical grain:
-- One row represents one combination of:
-- month + payment method.
--
-- ============================================================

SELECT
    d.month,
    f.method,
    SUM(f.amount) AS paid,
    COUNT(*) AS payments

FROM data_lab.shop.fact_payments f

JOIN data_lab.shop.dim_date d
    ON d.date = f.date

GROUP BY
    d.month,
    f.method

ORDER BY
    d.month,
    paid DESC;

month,method,paid,payments
2026-07,картка,341165,19
2026-07,готівка,212260,8
2026-07,Apple Pay,156915,9
2026-08,Apple Pay,305360,12
2026-08,картка,278955,14
2026-08,готівка,195940,8
2026-09,картка,39305,3
2026-09,Apple Pay,12775,1


In [0]:
%sql

-- ============================================================
-- Step 10.3. Orders that are not fully paid
--
-- Objective:
-- Identify orders where the total amount of payments is less
-- than the total sales amount of the order.
--
-- IMPORTANT:
-- The two fact tables are aggregated to the order level
-- BEFORE they are joined.
--
-- Why?
-- fact_sales has one row per product line.
-- fact_payments has one row per payment.
--
-- Joining the two raw fact tables directly by order_id can
-- multiply rows and therefore incorrectly increase SUM(amount).
--
-- LEFT JOIN is used between ordered and paid so that orders
-- with no payments at all are also included in the result.
--
-- ============================================================

WITH ordered AS (

    -- --------------------------------------------------------
    -- Aggregate sales to one row per order.
    --
    -- `ordered` = total sales amount of the order.
    -- --------------------------------------------------------

    SELECT
        order_id,
        customer_id,
        SUM(amount) AS ordered

    FROM data_lab.shop.fact_sales

    GROUP BY
        order_id,
        customer_id
),

paid AS (

    -- --------------------------------------------------------
    -- Aggregate payments to one row per order.
    --
    -- `paid`     = total amount already paid.
    -- `payments` = number of payment transactions.
    -- --------------------------------------------------------

    SELECT
        order_id,
        SUM(amount) AS paid,
        COUNT(*) AS payments

    FROM data_lab.shop.fact_payments

    GROUP BY
        order_id
)

-- ------------------------------------------------------------
-- Compare ordered amount with paid amount.
-- ------------------------------------------------------------

SELECT
    o.order_id,
    c.customer_name,
    c.city,

    -- Total value of products in the order
    o.ordered,

    -- Total amount already paid.
    -- If there are no payments, return 0 instead of NULL.
    COALESCE(p.paid, 0) AS paid,

    -- Outstanding amount:
    -- ordered amount minus amount already paid.
    o.ordered - COALESCE(p.paid, 0) AS debt,

    -- Number of payment transactions.
    -- If there are no payments, return 0.
    COALESCE(p.payments, 0) AS payments

FROM ordered o

-- LEFT JOIN is important:
-- an order without any payment must remain in the result.
LEFT JOIN paid p
    ON p.order_id = o.order_id

-- Customer attributes come from the shared dimension.
JOIN data_lab.shop.dim_customer c
    ON c.customer_id = o.customer_id

-- Keep only orders where the amount paid is less
-- than the total order amount.
WHERE COALESCE(p.paid, 0) < o.ordered

ORDER BY
    debt DESC;

order_id,customer_name,city,ordered,paid,debt,payments
1019,Марія Бондаренко,Одеса,81900,40950,40950,1
1041,Андрій Коваленко,Львів,23400,0,23400,0
1023,Ірина Лисенко,Вінниця,14850,0,14850,0
1006,Наталія Мельник,Дніпро,21000,6300,14700,1
1007,Сергій Поліщук,Київ,5600,0,5600,0
1040,Наталія Мельник,Дніпро,21000,15750,5250,2
1044,Андрій Коваленко,Львів,12500,8125,4375,2


In [0]:
%sql

-- ============================================================
-- Step 10.4. Debt by city
--
-- Objective:
-- Compare the total value of orders with the total amount
-- already paid for each city.
--
-- IMPORTANT:
-- Both fact tables are first aggregated to the order level.
-- Only after that are they joined.
--
-- This prevents fan-out / row multiplication when an order
-- contains multiple sales lines and multiple payments.
--
-- Result:
--   ordered = total value of orders
--   paid    = total amount already paid
--   debt    = outstanding amount
--
-- ============================================================

SELECT
    c.city,

    -- Total value of orders in the city.
    SUM(s.ordered) AS ordered,

    -- Total amount already paid.
    -- Orders without payments contribute 0.
    SUM(COALESCE(p.paid, 0)) AS paid,

    -- Outstanding debt:
    -- total ordered amount minus total paid amount.
    SUM(s.ordered) - SUM(COALESCE(p.paid, 0)) AS debt

FROM (

    -- --------------------------------------------------------
    -- Step 1. Aggregate sales to the order level.
    --
    -- One row = one order.
    -- --------------------------------------------------------

    SELECT
        order_id,
        customer_id,
        SUM(amount) AS ordered

    FROM data_lab.shop.fact_sales

    GROUP BY
        order_id,
        customer_id

) s

LEFT JOIN (

    -- --------------------------------------------------------
    -- Step 2. Aggregate payments to the order level.
    --
    -- One row = one order.
    -- --------------------------------------------------------

    SELECT
        order_id,
        SUM(amount) AS paid

    FROM data_lab.shop.fact_payments

    GROUP BY
        order_id

) p

    -- --------------------------------------------------------
    -- Step 3. Match payments to their orders.
    -- --------------------------------------------------------

    ON p.order_id = s.order_id

-- ------------------------------------------------------------
-- Step 4. Get the customer's city from the shared dimension.
-- ------------------------------------------------------------

JOIN data_lab.shop.dim_customer c
    ON c.customer_id = s.customer_id

-- ------------------------------------------------------------
-- Step 5. Aggregate the order-level results by city.
-- ------------------------------------------------------------

GROUP BY
    c.city

-- Show cities with the largest outstanding debt first.
ORDER BY
    debt DESC;

city,ordered,paid,debt
Одеса,453250,412300,40950
Львів,303050,275275,27775
Дніпро,336050,316100,19950
Вінниця,39650,24800,14850
Київ,328000,322400,5600
Харків,191800,191800,0


## Project Summary

### Objective

The project transforms raw CRM/POS CSV files into a structured
analytical data model in Databricks Lakehouse.

### Source Data

Two source files were used:

- `sales.csv` — product-level sales data
- `payments.csv` — payment transaction data

Source files are stored in a Unity Catalog Volume:

`/Volumes/data_lab/shop/raw/`

### Data Processing

The project includes:

1. Raw CSV ingestion
2. Source schema inspection
3. Data quality validation
4. Business key and grain identification
5. Dimension construction
6. Fact table construction
7. Referential integrity validation
8. Analytical SQL queries

### Analytical Data Model

The final model consists of:

- `dim_date`
- `dim_customer`
- `dim_product`
- `fact_sales`
- `fact_payments`

The model follows a Star Schema approach.

### Fact Table Grain

`fact_sales`:
one row represents one product line within an order.

`fact_payments`:
one row represents one payment transaction.

### Important Modeling Principle

The two fact tables are not joined directly for aggregation.

When sales and payments need to be compared, each fact table
is first aggregated to the same business grain, such as `order_id`.

This prevents fan-out and double counting.

### Analytical Results

The model supports analysis of:

- revenue by city, month and product category
- payments by month and payment method
- partially paid orders
- outstanding debt by city

### Technology Stack

- Databricks
- Apache Spark / PySpark
- SQL
- Delta Lake
- Unity Catalog
- Lakehouse architecture